# Solution A2 · OneHotEncoder from scratch: unknown categories and `drop`

**Type**: Implement from scratch

## Objective

Implement a one-hot encoder that learns the categories on the training set, always produces the same columns, handles
never-seen categories and supports `drop="first"`, checking it against `sklearn.preprocessing.OneHotEncoder`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = ds.loans(with_missing=False)
cat = df[["city", "contract_type"]].to_numpy(dtype=object)
cat_tr, cat_te = train_test_split(cat, test_size=0.25, random_state=0)
cat_te_clean = cat_te.copy()
cat_te = cat_te.copy()
cat_te[:5, 0] = "city_XX"                          # a city never seen in training

## Commented solution

In [ ]:
class MyOneHot:
    def __init__(self, drop=None):
        self.drop = drop

    def fit(self, X):
        self.categories_ = [np.unique(X[:, j]) for j in range(X.shape[1])]
        return self

    def transform(self, X):
        blocks = []
        for j, cat_j in enumerate(self.categories_):
            if self.drop == "first" and (~np.isin(X[:, j], cat_j)).any():
                raise ValueError(f"unknown category in feature {j}: not allowed with drop='first'")
            block = (X[:, j].reshape(-1, 1) == cat_j.reshape(1, -1)).astype(float)     # broadcasting
            blocks.append(block[:, 1:] if self.drop == "first" else block)
        return np.hstack(blocks)

    def names(self, feature_names):
        return [f"{name}_{c}" for name, cats in zip(feature_names, self.categories_)
                for c in (cats[1:] if self.drop == "first" else cats)]

In [ ]:
mine = MyOneHot().fit(cat_tr)
ref = OneHotEncoder(sparse=False, handle_unknown="ignore").fit(cat_tr)
mine_d = MyOneHot(drop="first").fit(cat_tr)
ref_d = OneHotEncoder(sparse=False, drop="first").fit(cat_tr)
print("shape on the test set:", mine.transform(cat_te).shape, "| with drop='first':", mine_d.transform(cat_te_clean).shape)
print("first row of the test set (unknown city):", mine.transform(cat_te)[0, :12])

## Check

In [ ]:
with verify():
    check(close(mine.transform(cat_tr), ref.transform(cat_tr)), "matches scikit-learn on the training set",
          "the columns must follow the order (feature, then sorted categories)")
    check(close(mine.transform(cat_te), ref.transform(cat_te)), "matches scikit-learn on the test set with the unknown city",
          "a never-seen category must give zeros in all the columns of that feature")
    check(mine.transform(cat_te)[:5, :12].sum() == 0, "the rows with the unknown city have 12 zeros in the city columns",
          "check the first five rows of the test set")
    check(list(mine.names(["city", "contract_type"])) == list(ref.get_feature_names_out(["city", "contract_type"])),
          "the column names match", "name = feature + '_' + category")
    check(close(mine_d.transform(cat_tr), ref_d.transform(cat_tr)) and mine_d.transform(cat_tr).shape[1] == 14,
          "drop='first' removes one column per feature (16 - 2 = 14)", "with drop='first' 14 columns remain")
    check(close(mine_d.transform(cat_te_clean), ref_d.transform(cat_te_clean)), "drop='first' matches on the clean test set", "review drop")
    try:
        mine_d.transform(cat_te)
        raised = False
    except ValueError:
        raised = True
    check(raised, "drop='first' with an unknown category raises ValueError",
          "with drop='first' the row of zeros would be ambiguous: raise ValueError")
done()

## What you learned

- The columns are decided once and for all by `fit` on the training set. That is why `transform` always produces the same
  number of columns in the same order, even on a test set with different categories: it is the advantage of an encoder with
  state over `pd.get_dummies` (you will see it break in exercise E1).
- An unknown category has no column: the only sensible choice is the row of zeros ("none of the known
  categories") or an error. With `drop="first"` the row of zeros would mean "first category": ambiguous, and that is
  why scikit-learn (1.1) forbids the combination.
- `drop="first"` does not change the predictions of a linear model with intercept, but it changes the **meaning** of the
  coefficients: they are differences with respect to the dropped category (you will see it in D1).

## Common pitfall

Building the columns by looking at the test set too (for example `np.unique` on the union). With real data the test set often contains
categories absent from the training set: the columns must come only from the training set, otherwise in production you will have
a different number of columns from the one the model was trained with.

## Interview question

> *Click on a question to show its answer.*

<details><summary>A one-hot in production gives a dimension error after a new city shows up. What happened and how do you prevent it?</summary>

Probably the columns were built with `pd.get_dummies` on the incoming data, so the number of columns
depends on the categories present: a new city adds a column and the model, trained on a fixed number,
refuses. I use an encoder with state (`OneHotEncoder(handle_unknown="ignore")`) inside the Pipeline: the columns are
fixed by the training set and the new categories produce zeros. In addition I monitor their frequency as a drift signal.

</details>